# 00 — System Checkpoint

**Purpose**: Run this notebook to see the current state of the ingestion system in ~2 minutes.  
It checks test suite status, artifact existence, source registry, and CLI health.  
It does NOT run the agent pipeline or make network calls.

**When to run**: At the start of any session, after restructuring, or to confirm Phase 2 gate criteria.

In [ ]:
# Setup
import subprocess
import json
import logging
from pathlib import Path
import yaml

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)-8s %(name)s: %(message)s'
)
logger = logging.getLogger('checkpoint')

ROOT = Path('.').resolve()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print(f'[SETUP] Project root: {ROOT}')
print('✓ Setup complete')

## Phase Status Table

In [ ]:
print('[STAGE] Phase Status Table ...')
phases = [
    ('Phase 1A', 'Source registry bootstrap', '✅ Complete', 'data/raw/mutual_funds/source_registry/'),
    ('Phase 1B', 'Provider website profiling', '✅ Complete', 'data/raw/mutual_funds/provider_profiles/'),
    ('Task-URL Agent', 'Ingestion pipeline (17 DB tables)', '✅ Substantially complete', 'mutual_fund_ingestion/agent/runner.py'),
    ('Test Coverage', 'Epics G-Q', '⚠️ In progress', 'tests/'),
    ('Phase 2', 'Document discovery', '❌ Not started', '—'),
]
header = f"{'Phase':<18} {'What it does':<40} {'Status':<25} {'Key output'}"
print(header)
print('-' * len(header))
for phase, desc, status, output in phases:
    print(f'{phase:<18} {desc:<40} {status:<25} {output}')
print('✓ Phase status displayed')

## Test Suite Status

In [ ]:
print('[STAGE] Running test suite ...')
result = subprocess.run(
    ['./financial_env/bin/python', '-m', 'pytest', 'tests/', '-q', '--tb=no'],
    capture_output=True, text=True, cwd=str(ROOT)
)
output = result.stdout + result.stderr
# Parse summary line
summary = [l for l in output.splitlines() if 'passed' in l or 'failed' in l or 'error' in l]
print('Test output:', summary[-1] if summary else 'No summary found')
print('Return code:', result.returncode)
print('✓ Test suite check complete')

## Artifact Existence Check

In [ ]:
print('[STAGE] Checking expected output artifacts ...')
artifacts = [
    ROOT / 'configs' / 'amc_sources.yaml',
    ROOT / 'tests' / 'fixtures' / 'provider_static.html',
    ROOT / 'tests' / 'fixtures' / 'amfi_members.html',
    ROOT / 'mutual_fund_ingestion' / 'agent' / 'runner.py',
    ROOT / 'mutual_fund_ingestion' / 'agent' / 'db.py',
]
for path in artifacts:
    status = '✅' if path.exists() else '❌ MISSING'
    print(f'  {status}  {path.relative_to(ROOT)}')

# Optional: check data artifacts (may not exist locally)
data_artifacts = [
    ROOT / 'data' / 'raw' / 'mutual_funds' / 'source_registry',
    ROOT / 'data' / 'raw' / 'mutual_funds' / 'provider_profiles',
]
print('\nData artifacts (may not exist in dev):')
for path in data_artifacts:
    status = '✅' if path.exists() else '⚠️  Not present (run pipeline first)'
    print(f'  {status}  {path.relative_to(ROOT)}')
print('✓ Artifact check complete')

## Source Registry Summary

In [ ]:
print('[STAGE] Loading source registry ...')
yaml_path = ROOT / 'configs' / 'amc_sources.yaml'
with open(yaml_path) as f:
    registry = yaml.safe_load(f)
sources = registry.get('sources', [])
enabled = [s for s in sources if s.get('enabled', True)]
by_role = {}
for s in sources:
    role = s.get('source_role', 'unknown')
    by_role[role] = by_role.get(role, 0) + 1
print(f'Total sources: {len(sources)}')
print(f'Enabled: {len(enabled)}')
for role, count in sorted(by_role.items()):
    print(f'  {role}: {count}')
print('✓ Source registry summary complete')

## CLI Smoke Test

In [ ]:
print('[STAGE] Running CLI help smoke test ...')
result = subprocess.run(
    ['./financial_env/bin/python', '-m', 'mutual_fund_ingestion', '--help'],
    capture_output=True, text=True, cwd=str(ROOT)
)
print('Exit code:', result.returncode)
# Show first 10 lines of help
for line in result.stdout.splitlines()[:10]:
    print(' ', line)
assert result.returncode == 0, f'CLI --help failed: {result.stderr}'
print('✓ CLI smoke test passed')

## Phase 2 Gate Check

In [ ]:
print('[STAGE] Phase 2 gate criteria check ...')
gates = [
    ('Test count ≥ 145', False, 'Current: 122 passed — need 23 more tests'),
    ('runner.py refactored (< 300 lines)', False, 'Current: 821 lines — see docs/04_in_progress/REFACTOR_runner.md'),
    ('Epic G validation tests complete', False, 'Pending: G001-G006'),
    ('Epic H discovery tests complete', False, 'Pending: H001-H005'),
    ('Epic P portfolio tests complete', False, 'Pending: P002-P003'),
    ('Epic N NAV tests complete', False, 'Pending: N001-N004'),
]
all_pass = True
for criterion, passing, note in gates:
    icon = '✅' if passing else '❌'
    print(f'  {icon} {criterion}')
    if not passing:
        print(f'     → {note}')
        all_pass = False
print()
print('Phase 2 gate:', '✅ OPEN' if all_pass else '❌ BLOCKED — complete above criteria first')
print('✓ Gate check complete')

## Assertions

In [ ]:
print('[STAGE] Running assertions ...')
# Source registry must have entries
assert len(sources) >= 50, f'Expected ≥50 registry sources, got {len(sources)}'
# configs file must exist
assert (ROOT / 'configs' / 'amc_sources.yaml').exists()
# agent module must be importable
import sys
sys.path.insert(0, str(ROOT))
from mutual_fund_ingestion.agent.config import AgentConfig
from mutual_fund_ingestion.agent.db import create_tables
print('✓ All assertions passed')

## Summary

In [ ]:
print('=== SYSTEM CHECKPOINT SUMMARY ===')
print()
print('🟢 Source registry:     OK  (≥50 sources in amc_sources.yaml)')
print('🟡 Test suite:          122 passed, 3 skipped (target: ≥145)')
print('🟡 Phase 2 gate:        BLOCKED — test coverage epics pending')
print('🟢 CLI:                 OK  (--help exits 0)')
print('🟡 runner.py:           821 lines — refactor needed before Phase 2')
print()
print('Next action: TASK-B001 → docs/06_plans/active/BATCH_B_docs.md')
print('===================================')